# 03 | Data Preparation & Analytical Modeling

**Project:** Dubai Short-Term Rental Intelligence  
**Data provider:** AirROI Dubai (Kaggle distribution)  
**Coverage:** March 2021–February 2026  
**Tools:** Python · pandas · reproducible transformation script

## Analytical objective

Transform four independently reconciled source files into a **validated, nonduplicated analytical model** for historical comparisons, SQL analysis, property benchmarking, and Power BI reporting. The emphasis is on explicit data grain, reproducible transformations, and measurable quality gates.

**Scope:** Results describe the supplied sample of AirROI observations, **not** a full census of Dubai rentals. Occupancy and revenue are **provider estimates**, not verified transactions. The source is independent of Airbnb.

## 1. Source inputs and reproducibility

| Source file | Granularity | Role |
|:--|:--|:--|
| `past_rates.csv` | One listing per observed month | Primary historical observation table |
| `listings_monthly.csv` | Overlapping listing-month subset | Cross-check shared history; **do not append** |
| `past_rates.parquet` | Same historical listing-months | Reconcile all shared fields; no extra rows |
| `listings.parquet` | Undated full listing snapshot and nested months | Broader descriptive metadata and separate native booked-rate metric |

The canonical transformations reside in [`src/01_validate_and_prepare.py`](../src/01_validate_and_prepare.py). This notebook runs that script and independently audits its outputs. Raw inputs are read from `data/raw/` and never modified. Generated data and audit artifacts are written to `data/processed/` and `reports/`.

**Reproduction:** Install the dependencies from `requirements.txt`, place all four original source files in `data/raw/`, and run all cells from the project root or `notebooks/`. Source records are not redistributed in the public repository pending license verification.

**Snapshot control (independent audit, 10 October 2026):** Fixed figures in this narrative refer to the four original files identified by SHA-256 in `reports/source_audit.json`. Executable tables derive values at runtime. Review the narrative if replacing the source snapshot. Raw data and listing-level outputs remain private; conclusions describe this sample and provider estimates.

In [1]:
from pathlib import Path
import runpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CWD = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (CWD, CWD.parent, CWD.parent.parent)
                     if (path / "README.md").is_file() and (path / "data" / "raw").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the project root or notebooks/ directory.")

PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports" / "data_preparation"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 14)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Project:", PROJECT_ROOT.name)
print("Pipeline:", (PROJECT_ROOT / "src" / "01_validate_and_prepare.py").relative_to(PROJECT_ROOT))

## 2. Transformation methodology

| Decision | Implementation | Analytical justification |
|:--|:--|:--|
| **Avoid double-counting** | Retain `past_rates.csv` as the full listing-month history; use the CSV enrichment for cross-checking and the full Parquet snapshot for descriptive metadata | The enriched rows overlap the primary history |
| **Define stable keys** | Preserve `listing_id` as **text** and standardize `month_date` to the month start | Protect long identifiers and enforce a consistent grain |
| **Preserve missing information** | Keep unavailable neighborhood and bedroom data as missing; do not infer or silently fill | Prevent fabricated segmentation and misleading comparables |
| **Build historical fact** | One row per `listing_id × month_date` with rates and provider-estimated metrics | Enable monthly SQL / BI measures without duplicate observations |
| **Build listing dimension** | One row per listing, combining the last known primary metadata with undated Parquet snapshot attributes | Make property descriptors accessible, with **snapshot** limitations documented |
| **Build month dimension** | Consecutive monthly calendar covering the observed period | Support month sorting and time comparisons |
| **Track sample coverage** | Capture counts, metadata missingness, changes and adjacent-month listing matches | Separate coverage shifts from possible economic changes |
| **Validate source overlap** | Reject duplicate keys or disagreements in duplicated source measures | Detect ingestion errors before building analytical tables |

**Temporal limitation:** The listing dimension contains the **latest available** descriptors, not point-in-time property history. Historic competitor matching must use month-level fact fields where available; static dimension attributes must not be assumed constant in the past.

In [2]:
pipeline = runpy.run_path(str(PROJECT_ROOT / "src" / "01_validate_and_prepare.py"))["run"]
source_audit = pipeline()

source_summary = pd.DataFrame([
    {"Measure": "Primary listing-month observations", "Value": f"{source_audit['primary_rows']:,}"},
    {"Measure": "Enriched overlapping observations", "Value": f"{source_audit['enriched_rows']:,}"},
    {"Measure": "Unique historical listings", "Value": f"{source_audit['unique_historical_listings']:,}"},
    {"Measure": "Reporting months", "Value": source_audit['month_count']},
    {"Measure": "Historical period", "Value": f"{source_audit['first_month']} to {source_audit['latest_month']}"},
    {"Measure": "Enriched keys outside primary data", "Value": source_audit['enriched_rows_not_in_primary']},
    {"Measure": "Source fields with any disagreement", "Value": sum(v > 0 for v in source_audit['shared_field_mismatches'].values())},
])
display(source_summary)

## 3. Resulting analytical model

| Table | Grain | Intended use |
|:--|:--|:--|
| `fact_listing_month` | Listing × observed month | Historical rates, estimated occupancy and property-month outcomes |
| `dim_listing` | One record per listing | Last-known descriptors; enriched fields available for a subset |
| `dim_month` | One record per month | Chronological comparisons and calendar filters |
| `fact_coverage_month` | One record per month | Source coverage and changes in sample composition |

**Model relationships:** `fact_listing_month.listing_id → dim_listing.listing_id` and `fact_listing_month.month_date → dim_month.month_date`. The coverage table relates to `dim_month` by `month_date`, but must not be directly joined to the listing fact without respecting its different grain.

In [3]:
fact = pd.read_csv(PROCESSED / "fact_listing_month.csv",
                   dtype={"listing_id": "string"}, parse_dates=["month_date"])
listing = pd.read_csv(PROCESSED / "dim_listing.csv",
                      dtype={"listing_id": "string", "host_id": "string"})
months = pd.read_csv(PROCESSED / "dim_month.csv", parse_dates=["month_date"])
coverage = pd.read_csv(PROCESSED / "fact_coverage_month.csv", parse_dates=["month_date"])

model_inventory = pd.DataFrame([
    {"Table": "fact_listing_month", "Rows": len(fact), "Grain": "Listing × month",
     "Duplicate keys": int(fact.duplicated(["listing_id", "month_date"]).sum())},
    {"Table": "dim_listing", "Rows": len(listing), "Grain": "Listing",
     "Duplicate keys": int(listing.listing_id.duplicated().sum())},
    {"Table": "dim_month", "Rows": len(months), "Grain": "Month",
     "Duplicate keys": int(months.month_date.duplicated().sum())},
    {"Table": "fact_coverage_month", "Rows": len(coverage), "Grain": "Month",
     "Duplicate keys": int(coverage.month_date.duplicated().sum())},
])
display(model_inventory)

## 4. Independent quality gates

A successful transformation is not sufficient on its own. The following checks assert **unique and non-null keys**, complete referential integrity, consecutive reporting months, reconciliation with the source records, valid currency and reporting dates, and basic numerical plausibility. The checks intentionally distinguish **missing optional attributes** from **invalid required measures**.

A passing validation establishes structural consistency in the fields tested—not external accuracy of the provider's estimated business metrics.

In [4]:
month_sequence = pd.date_range(months.month_date.min(), months.month_date.max(), freq="MS")
expected_calendar = pd.Series(month_sequence).reset_index(drop=True)
actual_calendar = months.month_date.sort_values().reset_index(drop=True)

required_money = ["native_avg_daily_rate", "native_revenue", "native_revpar"]
financial_fields_complete = fact[required_money].notna().all().all()
financial_fields_nonnegative = fact[required_money].ge(0).all().all()
valid_estimated_occupancy = fact.occupancy.notna().all() and fact.occupancy.between(0, 1).all()

checks = {
    "No duplicate keys across all analytical tables": model_inventory["Duplicate keys"].eq(0).all(),
    "No null primary / dimension identifiers": (
        fact[["listing_id", "month_date"]].notna().all().all()
        and listing.listing_id.notna().all()
        and months.month_date.notna().all()
        and coverage.month_date.notna().all()
    ),
    "Every fact listing has a dimension row": fact.listing_id.isin(listing.listing_id).all(),
    "Every fact month has a calendar row": fact.month_date.isin(months.month_date).all(),
    "Every coverage month has a calendar row": coverage.month_date.isin(months.month_date).all(),
    "Calendar is continuous and month-start aligned": actual_calendar.equals(expected_calendar),
    "Fact dates are month-start aligned": fact.month_date.dt.is_month_start.all(),
    "Monthly coverage reconciles to fact observations": (
        int(coverage.observed_listings.sum()) == len(fact)
        and len(coverage) == len(months)
    ),
    "Fact count equals validated primary source": len(fact) == source_audit["primary_rows"],
    "No enriched keys missing in primary history": source_audit["enriched_rows_not_in_primary"] == 0,
    "No disagreement in overlapping source fields": not any(source_audit["shared_field_mismatches"].values()),
    "Listing IDs are preserved as strings": pd.api.types.is_string_dtype(fact.listing_id),
    "Reported currency is AED for every observation": fact.currency.eq("AED").all(),
    "Days reconcile to calendar month length": (
        fact.available_days.add(fact.unavailable_days).eq(fact.month_date.dt.days_in_month).all()
    ),
    "Monetary measures are present": financial_fields_complete,
    "Monetary measures are nonnegative": financial_fields_nonnegative,
    "Estimated occupancy is within 0 to 1": valid_estimated_occupancy,
}
quality_gates = pd.DataFrame([
    {"Check": label, "Status": "PASS" if bool(passed) else "FAIL"}
    for label, passed in checks.items()
])
display(quality_gates)
quality_gates.to_csv(REPORT_DIR / "03_model_quality_gates.csv", index=False)
if not all(checks.values()):
    raise AssertionError("One or more analytical model quality gates failed.")
print(f"Quality gates passed: {len(checks)}/{len(checks)}")

## 5. Historical sample coverage and comparability

The number of included listings changes across months. Unadjusted count changes must **not** be interpreted as expansion or contraction of the whole Dubai short-term rental market. When analyzing time trends, show monthly source coverage; for property-level changes, prioritize the same listings observed across the periods being compared.

In [5]:
coverage_review = coverage[[
    "month_date", "observed_listings", "active_listings",
    "coverage_change_pct", "same_listing_prev_month"
]].copy()
material_changes = coverage_review.loc[coverage_review.coverage_change_pct.abs() > 25].copy()

print("Months with >25% absolute change in observed listings:", len(material_changes))
display(material_changes.sort_values("month_date"))
print("Latest reporting periods:")
display(coverage_review.tail(6))

assert int(coverage.observed_listings.sum()) == len(fact)
assert int(fact.same_listing_prev_month.sum()) == int(coverage.same_listing_prev_month.sum())

fig, ax = plt.subplots(figsize=(10.8, 4.2))
ax.plot(coverage.month_date, coverage.observed_listings, linewidth=2, label="Observed listings")
ax.plot(coverage.month_date, coverage.active_listings, linewidth=2, label="Provider-labelled active listings")
ax.set(title="AirROI Dubai sample coverage by reporting month",
       ylabel="Unique listings", xlabel="Reporting month")
ax.legend(frameon=False)
ax.grid(alpha=0.15)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "03_sample_coverage.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Metadata readiness and treatment of missing values

The `dim_listing` table uses `listings.parquet` for all source IDs. The source is an undated snapshot; individual fields can still be missing. This has direct consequences for segmentation and matching coverage. A listing **without enriched metadata is not removed from the historical fact**; it remains eligible for analyses based on the available fact fields. No neighborhoods, bedrooms, guest capacities, or estimated outcomes are artificially filled.

In [6]:
enriched_count = int(listing.enriched_metadata_available.eq(True).sum())
enriched_share = enriched_count / len(listing)
latest_date = fact.month_date.max()
latest = fact.loc[fact.month_date.eq(latest_date)]

metadata_audit = pd.DataFrame([
    {"Metric": "Historical listings with enriched metadata", "Value": f"{enriched_count:,} / {len(listing):,}"},
    {"Metric": "Share with enriched metadata", "Value": f"{enriched_share:.1%}"},
    {"Metric": "Latest observed properties", "Value": f"{len(latest):,}"},
    {"Metric": "Missing neighborhood (latest month)", "Value": f"{latest.neighborhood.isna().mean():.1%}"},
    {"Metric": "Missing bedrooms (latest month)", "Value": f"{latest.bedrooms.isna().mean():.1%}"},
    {"Metric": "Missing ADR (all months)", "Value": f"{fact.native_avg_daily_rate.isna().sum():,}"},
    {"Metric": "Missing estimated occupancy (all months)", "Value": f"{fact.occupancy.isna().sum():,}"},
])
display(metadata_audit)
metadata_audit.to_csv(REPORT_DIR / "03_metadata_readiness.csv", index=False)

# Preserve the meaning of source fields: no inferred income, booking counts, or locations.
assert enriched_count == source_audit["snapshot_enriched_unique_listings"]
print("Validated output artifacts:")
for artifact in sorted(REPORT_DIR.glob("03_*.csv")):
    print("-", artifact.relative_to(PROJECT_ROOT))

## 7. Key findings and analytical decisions

The preparation pipeline produced **99,745 distinct listing-month observations**, **4,364 listing dimension records**, **60 reporting months**, and **60 monthly coverage records**. The **22,629 enriched observations were not added** to the fact table because they overlap the primary historical data. The quality gates above establish that these tables are structurally consistent for the fields tested.

**Sample volatility.** Five months show a change of more than 25% in the number of observed listings. Notably, coverage fell **34.75% in October 2025** and rose **50.40% in November 2025**. The dashboard must display observed-listing counts alongside historical performance and avoid treating these changes as market growth or decline.

**Metadata limitations.** The enriched CSV covers 1,000 listings, but the audited Parquet snapshot supplies metadata for **all 4,364**. Ratings, beds and capacity remain partially missing; presence is not completeness. In the latest observed month, approximately **77.7%** of records have no neighborhood and **12.1%** have no bedroom value. Analyses using such descriptors require explicit eligible-sample counts; optional attributes remain missing rather than being guessed.

**Downstream modeling decisions**

- **SQL and Power BI:** Use `fact_listing_month` for time-varying measures, `dim_listing` for latest-known metadata, `dim_month` for calendar filtering, and `fact_coverage_month` for source coverage. Avoid merging monthly aggregates into the listing-level fact without respecting grain.
- **Competitor benchmarking:** Match listings using **validated month-level** coordinates, room type, and bedroom count where available; do not use undated guest-capacity or quality metadata for historical matching. Never assign unsupported benchmarks to insufficiently comparable properties.
- **Metric interpretation:** All amounts are recorded in **AED**. Treat `occupancy`, `native_revenue`, and `native_revpar` as **AirROI/provider estimates** and avoid claims about confirmed bookings, actual profits, or population-wide market activity.
- **Auditability:** Transformation logic is version-controlled in `src/01_validate_and_prepare.py`; the notebook saves quality-gate and metadata-coverage tables for review.

**Next analysis:** [`04_competitor_benchmarking.ipynb`](04_competitor_benchmarking.ipynb) evaluates peer-group construction and benchmark reliability before those outputs enter the Power BI model.